### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Building a recommendation model is 20% of the work. The other 80% is the SYSTEM: feature pipelines, serving infrastructure, A/B testing, monitoring for feedback loops, and handling cold start. A mediocre algorithm in a great system beats a great algorithm in a mediocre system every time.

**Mental Model:** Think of a restaurant. The CHEF (ML model) is important, but so is: the supply chain (data pipeline), waiters (serving layer), menu design (candidate generation), quality control (monitoring), and customer feedback cards (A/B testing). A restaurant fails because of bad service more often than bad food.

**Common Junior Pitfall:** Ignoring feedback loops. Users only interact with items you RECOMMEND. If your model keeps recommending popular items, it gets MORE data on popular items, making it recommend popular items even more. This creates a filter bubble that reduces diversity and user satisfaction.

---

## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · ANN Retrieval at Scale](#1-ann-retrieval-at-scale)
  * [The Problem](#the-problem)
  * [Approximate Nearest Neighbor (ANN)](#approximate-nearest-neighbor-ann)
* [2 · Feature Store Integration](#2-feature-store-integration)
  * [Real-Time Features Make the Difference](#real-time-features-make-the-difference)
* [3 · Online Learning & Multi-Armed Bandits](#3-online-learning-multi-armed-bandits)
  * [The Explore-Exploit Tradeoff in RecSys](#the-explore-exploit-tradeoff-in-recsys)
* [4 · Cold Start & Diversity](#4-cold-start-diversity)
  * [Cold Start Strategies](#cold-start-strategies)
  * [Diversity: Breaking the Filter Bubble](#diversity-breaking-the-filter-bubble)
* [5 · End-to-End System Design](#5-end-to-end-system-design)
  * [Production RecSys Architecture](#production-recsys-architecture)
  * [🎓 DEEP QUESTION ANSWERED](#deep-question-answered)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Latency budget](#q1-latency-budget)
  * [Q2: Feedback loops](#q2-feedback-loops)
  * [Q3: System design](#q3-system-design)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: A/B Test Analysis](#exercise-1-ab-test-analysis)
  * [Exercise 2: Session-Based Recommendations](#exercise-2-session-based-recommendations)
  * [Exercise 3: Full Pipeline](#exercise-3-full-pipeline)
* [🎉 RS Track Complete!](#rs-track-complete)


In [ ]:
!pip install -q numpy matplotlib

## 1 · ANN Retrieval at Scale

### The Problem

You have the user embedding (32-dim vector). You need to find the top 100 most relevant items from 100 MILLION item embeddings. Brute-force cosine similarity would take ~1 second. You need <10ms.

### Approximate Nearest Neighbor (ANN)

ANN algorithms trade a tiny amount of accuracy for massive speed:

| Algorithm | Speed | Recall@100 | Memory | Best For |
|-----------|-------|-----------|--------|----------|
| **HNSW** | ~1ms per query | ~99% | High (graph) | Highest quality |
| **IVF** | ~2ms per query | ~95% | Medium (clusters) | Large-scale |
| **ScaNN** | ~0.5ms per query | ~98% | Medium | Google's choice |
| **Product Quantization** | ~0.3ms per query | ~90% | Very low | Memory-constrained |

These are the SAME algorithms used in vector databases (NB27) for RAG retrieval.

```python
# Production retrieval with FAISS (Meta)
import faiss

# Build index (offline)
index = faiss.IndexHNSWFlat(embed_dim, 32)  # HNSW with M=32
index.add(all_item_embeddings)  # Add 100M items

# Query (online, <5ms)
distances, indices = index.search(user_embedding, k=100)  # Top 100
```

In [ ]:
import numpy as np
import time

class SimpleANN:
    """Simplified ANN retrieval using IVF-like approach.
    
    Production: use FAISS, ScaNN, or vector databases (NB27).
    """
    
    def __init__(self, embeddings, n_clusters=10):
        self.embeddings = embeddings
        self.n_clusters = n_clusters
        
        # Step 1: Cluster item embeddings (offline)
        # Simplified k-means (production: use FAISS IVF)
        from sklearn.cluster import MiniBatchKMeans
        self.kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42)
        self.cluster_labels = self.kmeans.fit_predict(embeddings)
        
        # Build inverted index
        self.clusters = {}
        for i, label in enumerate(self.cluster_labels):
            if label not in self.clusters:
                self.clusters[label] = []
            self.clusters[label].append(i)
    
    def search(self, query, top_k=10, n_probes=3):
        """Find top-k nearest neighbors using IVF strategy.
        
        1. Find n_probes nearest clusters to query
        2. Only search within those clusters
        3. Much faster than brute force!
        """
        # Find nearest clusters
        cluster_dists = np.linalg.norm(self.kmeans.cluster_centers_ - query, axis=1)
        nearest_clusters = cluster_dists.argsort()[:n_probes]
        
        # Search only in nearest clusters
        candidates = []
        for c in nearest_clusters:
            candidates.extend(self.clusters.get(c, []))
        
        # Score candidates
        candidate_embeds = self.embeddings[candidates]
        scores = candidate_embeds @ query
        top_indices = scores.argsort()[::-1][:top_k]
        
        return [candidates[i] for i in top_indices], scores[top_indices]

# Benchmark: brute force vs ANN
n_items = 100_000
embed_dim = 32
item_embeddings = np.random.randn(n_items, embed_dim).astype(np.float32)
item_embeddings /= np.linalg.norm(item_embeddings, axis=1, keepdims=True)
query = np.random.randn(embed_dim).astype(np.float32)
query /= np.linalg.norm(query)

# Brute force
t0 = time.time()
brute_scores = item_embeddings @ query
brute_top = brute_scores.argsort()[::-1][:10]
t_brute = time.time() - t0

# ANN (IVF-style)
from sklearn.cluster import MiniBatchKMeans
ann = SimpleANN(item_embeddings, n_clusters=100)
t0 = time.time()
ann_top, ann_scores = ann.search(query, top_k=10, n_probes=5)
t_ann = time.time() - t0

# Compare
recall = len(set(brute_top) & set(ann_top)) / 10
print(f'Retrieval Benchmark ({n_items:,} items, {embed_dim}-dim):')
print(f'  Brute force: {t_brute*1000:.1f}ms')
print(f'  ANN (IVF):   {t_ann*1000:.1f}ms')
print(f'  Speedup:     {t_brute/t_ann:.1f}x')
print(f'  Recall@10:   {recall:.0%} (% of true top-10 found)')
print(f'\n  At YouTube scale (100M items): brute force ~10s, ANN ~5ms')

---
## 2 · Feature Store Integration

### Real-Time Features Make the Difference

The ranking model needs features that are FRESH — computed in real time:

| Feature | Freshness | Where Computed |
|---------|-----------|---------------|
| User demographics | Static | User DB |
| Item metadata | Daily | Content pipeline |
| User's last 10 interactions | Real-time (<1s) | **Feature Store** (NB08) |
| Item click-rate in last hour | Real-time (<5min) | **Streaming** (NB07) |
| User's session engagement | Real-time (<10s) | **Feature Store** |

```python
# Feast Feature Store integration (NB08)
from feast import FeatureStore
store = FeatureStore(repo_path='./feature_repo')

# At serving time: fetch real-time features for ranking
features = store.get_online_features(
    features=['user:click_count_1h', 'user:avg_session_time',
              'item:views_last_hour', 'item:trending_score'],
    entity_rows=[{'user_id': user_id, 'item_id': item_id}
                 for item_id in candidate_items]
).to_df()
```

**This is where NB07 (Streaming) and NB08 (Feature Store) connect to RecSys.**

---
## 3 · Online Learning & Multi-Armed Bandits

### The Explore-Exploit Tradeoff in RecSys

| Strategy | Behavior | Problem |
|----------|----------|--------|
| **Pure exploit** | Only recommend items with highest predicted score | Filter bubble, misses new content |
| **Pure explore** | Random recommendations | Terrible user experience |
| **Bandit** | Mostly exploit, strategically explore | Best of both worlds |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class ThompsonSamplingBandit:
    """Thompson Sampling for recommendation exploration.
    
    Used by: Spotify (Discover Weekly), Instagram (Explore page).
    
    Idea: maintain a Beta distribution for each item's click probability.
    Sample from the distribution (stochastic → explores uncertain items).
    Items with more data → tighter distribution → less exploration.
    New items → wide distribution → more exploration.
    """
    
    def __init__(self, n_items):
        # Beta distribution parameters for each item
        self.alpha = np.ones(n_items)  # Successes + 1 (prior)
        self.beta = np.ones(n_items)   # Failures + 1 (prior)
    
    def select(self, candidates, n=5):
        """Select n items to recommend from candidates."""
        # Sample click probability from each item's Beta distribution
        sampled_probs = np.random.beta(self.alpha[candidates], self.beta[candidates])
        # Select top-n by sampled probability
        top_n = sampled_probs.argsort()[::-1][:n]
        return [candidates[i] for i in top_n]
    
    def update(self, item, clicked):
        """Update beliefs based on user feedback."""
        if clicked:
            self.alpha[item] += 1  # More successes
        else:
            self.beta[item] += 1   # More failures

# Simulate: 50 items with hidden click probabilities
np.random.seed(42)
n_items = 50
true_click_rates = np.random.beta(2, 10, n_items)  # Most items have low CTR
true_click_rates[5] = 0.8   # Item 5 is amazing (hidden)
true_click_rates[42] = 0.7  # Item 42 is great (hidden)

# Run Thompson Sampling
bandit = ThompsonSamplingBandit(n_items)
cumulative_reward = [0]
choices_history = []

for step in range(1000):
    candidates = list(range(n_items))
    selected = bandit.select(candidates, n=5)
    
    # Simulate user interaction
    for item in selected:
        clicked = np.random.random() < true_click_rates[item]
        bandit.update(item, clicked)
        choices_history.append(item)
    
    reward = sum(true_click_rates[i] for i in selected) / 5
    cumulative_reward.append(cumulative_reward[-1] + reward)

# Compare to random baseline
random_reward = np.mean(true_click_rates) * 1000

plt.figure(figsize=(10, 5))
plt.plot(cumulative_reward[1:], lw=2, color='steelblue', label='Thompson Sampling')
plt.plot(np.linspace(0, random_reward, 1000), '--', color='gray', label='Random', lw=2)
optimal_reward = np.sort(true_click_rates)[-5:].mean() * 1000
plt.plot(np.linspace(0, optimal_reward, 1000), '--', color='green', alpha=0.5, label='Optimal', lw=2)
plt.xlabel('Step')
plt.ylabel('Cumulative Reward')
plt.title('Thompson Sampling: Explore-Exploit in RecSys')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Thompson Sampling quickly identifies best items (5, 42).')
print(f'Early: explores widely. Late: mostly exploits best items.')
print(f'\nLearned click rates for top items:')
learned = bandit.alpha / (bandit.alpha + bandit.beta)
top_learned = learned.argsort()[::-1][:5]
for i in top_learned:
    print(f'  Item {i}: learned CTR={learned[i]:.3f}, true CTR={true_click_rates[i]:.3f}')

---
## 4 · Cold Start & Diversity

### Cold Start Strategies

| Scenario | Strategy | Implementation |
|----------|----------|---------------|
| **New user, no history** | Popularity-based + onboarding | Show trending items, ask preferences |
| **New item, no interactions** | Content-based features | Use metadata embeddings (genre, description) |
| **New user + new item** | Global popularity | Most-popular baseline |

### Diversity: Breaking the Filter Bubble

| Technique | How It Works |
|-----------|-------------|
| **MMR** (Maximal Marginal Relevance) | Balance relevance and diversity in final results |
| **Category coverage** | Ensure recommendations span N different categories |
| **Exploration slots** | Reserve 10% of slots for exploratory (uncertain) items |
| **Temporal diversity** | Don't repeat recommendations from last session |

In [ ]:
import numpy as np

def mmr_diversify(item_scores, item_embeddings, top_k=10, lambda_param=0.5):
    """Maximal Marginal Relevance — balance relevance and diversity.
    
    At each step, select the item that maximizes:
    MMR(i) = λ * relevance(i) - (1-λ) * max_similarity(i, already_selected)
    
    λ=1.0: pure relevance (no diversity)
    λ=0.0: pure diversity (ignores relevance)
    λ=0.5: balanced
    """
    n_items = len(item_scores)
    selected = []
    remaining = list(range(n_items))
    
    # Normalize scores to [0, 1]
    scores_norm = (item_scores - item_scores.min()) / (item_scores.max() - item_scores.min() + 1e-8)
    
    for _ in range(top_k):
        best_score = -float('inf')
        best_item = None
        
        for item in remaining:
            relevance = scores_norm[item]
            
            # Max similarity to already selected items
            if selected:
                sims = [np.dot(item_embeddings[item], item_embeddings[s]) 
                        for s in selected]
                max_sim = max(sims)
            else:
                max_sim = 0
            
            mmr_score = lambda_param * relevance - (1 - lambda_param) * max_sim
            
            if mmr_score > best_score:
                best_score = mmr_score
                best_item = item
        
        selected.append(best_item)
        remaining.remove(best_item)
    
    return selected

# Demo: pure relevance vs MMR diversity
np.random.seed(42)
n_items = 20
scores = np.random.randn(n_items) + np.linspace(2, 0, n_items)  # Item 0 is most relevant
embeddings = np.random.randn(n_items, 8)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
# Make items 0-4 very similar to each other
embeddings[:5] = embeddings[0] + np.random.randn(5, 8) * 0.1
embeddings[:5] /= np.linalg.norm(embeddings[:5], axis=1, keepdims=True)

# Pure relevance (no diversity)
pure_top = scores.argsort()[::-1][:5]
print(f'Pure Relevance (top 5): {pure_top.tolist()}')
print(f'  All from the same cluster — boring and redundant!')

# MMR diversified
mmr_top = mmr_diversify(scores, embeddings, top_k=5, lambda_param=0.5)
print(f'\nMMR Diversified (top 5): {mmr_top}')
print(f'  Mix of relevant AND diverse items — better user experience!')

---
## 5 · End-to-End System Design

### Production RecSys Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    OFFLINE PIPELINE                          │
│                                                             │
│  [User Events] → [Kafka/NB07] → [Feature Pipeline]         │
│                                      ↓                     │
│                              [Feature Store/NB08]           │
│                                      ↓                     │
│  [Training Data] → [Model Training] → [Model Registry]     │
│                                      ↓                     │
│                    [Item Embedding Index (HNSW/NB27)]       │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    ONLINE SERVING (<100ms total)             │
│                                                             │
│  User Request                                               │
│       ↓                                                     │
│  [User Tower] → user embedding         (~5ms)               │
│       ↓                                                     │
│  [ANN Retrieval] → 1000 candidates     (~10ms)              │
│       ↓                                                     │
│  [Feature Fetch] from Feature Store    (~5ms)               │
│       ↓                                                     │
│  [Ranking Model] → scored candidates   (~20ms)              │
│       ↓                                                     │
│  [Re-Ranking] → diversity, business rules (~5ms)            │
│       ↓                                                     │
│  [Response] → 10-20 recommendations                         │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    MONITORING                                │
│                                                             │
│  [A/B Testing] → [CTR Tracking] → [Drift Detection/NB21]   │
│  [Feedback Loop Analysis] → [Diversity Metrics]             │
└─────────────────────────────────────────────────────────────┘
```

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How does TikTok serve 1B+ users in real time with new content every minute?*

**A:** A multi-layer architecture:
1. **Item embeddings are computed WHEN UPLOADED** — the item tower runs once per video, not per user.
2. **ANN index updates incrementally** — new items are added to the HNSW index without rebuilding.
3. **User embeddings update with each interaction** — streaming features from Kafka (NB07) feed into real-time user profiles.
4. **Models are trained continuously** — not retrained from scratch, but updated incrementally with new data.
5. **Bandits handle cold start** — new videos get initial exploration slots, Thompson Sampling quickly identifies viral content.

---

## ✅ Knowledge Check

### Q1: Latency budget
Your total serving budget is 100ms. Candidate generation takes 15ms, feature fetching takes 10ms, and re-ranking takes 5ms. What's the maximum latency for the ranking model?

<details><summary>👉 Answer</summary>

100 - 15 - 10 - 5 = 70ms for the ranking model. But in practice, you'd budget less (~40ms) to leave room for network overhead, serialization, and tail latency spikes (p99). This constraint limits model complexity — you might use a 2-layer MLP instead of a deep transformer.
</details>

### Q2: Feedback loops
Your RecSys keeps recommending pop music to everyone. Pop's click rate keeps going up. What's happening and how do you fix it?

<details><summary>👉 Answer</summary>

Popularity bias feedback loop: popular items get recommended → get more clicks → appear more popular → get recommended more. Fix: (1) Decouple exposure from clicks — normalize CTR by number of times shown, (2) Add exploration (bandits) to surface underexposed items, (3) Penalize already-popular items in the ranking score, (4) Monitor diversity metrics alongside CTR.
</details>

### Q3: System design
Design a recommendation system for a job board (100K jobs, 10M users). What's different from Netflix?

<details><summary>👉 Answer</summary>

Key differences: (1) Items expire quickly (jobs close), requiring frequent index updates, (2) Location is critical (geographic filtering before ANN), (3) Matching is bi-directional (job must also match candidate), (4) Success metric is APPLICATION, not just click (delayed, sparse reward), (5) Cold start is severe (new jobs posted daily, each with short lifespan). Use NB35's system design framework.
</details>

---

## 🔨 Practical Practice

### Exercise 1: A/B Test Analysis
Simulate an A/B test: model A has 5.2% CTR, model B has 5.5% CTR over 10,000 impressions each. Is B significantly better? Implement a statistical significance test (chi-squared or z-test).

### Exercise 2: Session-Based Recommendations
Implement a GRU-based session recommender: input is the sequence of items the user viewed in the current session, output is predicted next item. No user ID needed — pure session context.

### Exercise 3: Full Pipeline
Build a mini end-to-end RecSys: candidate generation (two-tower) → ranking (feature-rich model) → re-ranking (MMR diversity). Measure end-to-end latency.

---

## 🎉 RS Track Complete!

```
RS 01: Foundations          → Content-based & Collaborative Filtering
RS 02: Matrix Factorization → Learned embeddings from rating patterns
RS 03: Deep Learning        → Neural CF, Two-Tower, production funnel
RS 04: Production Systems   → ANN retrieval, bandits, diversity, system design
```

**Connections to the main curriculum:**

| RS Concept | Main Curriculum |
|-----------|----------------|
| Two-tower retrieval | NB27 (Vector DBs), NB28 (RAG) |
| Feature Store | NB08 (Feast) |
| Streaming features | NB07 (Kafka + Flink) |
| System design | NB35 (ML System Design) |
| ANN search (HNSW, IVF) | NB27 (Vector Databases) |
| Embeddings | NLP/02 (Word2Vec), NB27 |
| A/B testing | NB14 (Testing AI Applications) |
| Monitoring feedback loops | NB21 (Monitoring & Drift) |

**Next →** Return to the main track: NB35 (ML System Design) for full design practice.